# Scalar nonlocal AMB M00–M05 · ShellForce CNEEP

기존 `Corr_AMB2D_ShellForce.ipynb`의 **시뮬레이션 → 학습 → checkpoint → EPR/map/shell 비교** 흐름에 명세의 여섯 모델을 연결합니다. 원본 네트워크 파일은 수정하지 않습니다.

기본 실행은 CPU의 **짧은 실행 점검(smoke)** 입니다. 아래 `RUN_EXTENDED`, `DO_TRAIN` 설정으로 긴 물리 실험과 NN 학습을 선택합니다. 짧은 실행의 유한한 출력이나 loss 감소는 stationarity, 유의한 EPR, kernel recovery의 검증이 아닙니다. 외부 설치·네트워크 요청은 실행하지 않습니다.

- 상태는 보존 scalar `phi` 하나, `h=M=1`; 시뮬레이션과 EPR 누적은 float64입니다.
- 물리 kernel의 전체 periodic support를 유지하고 NN 반경 밖은 `tail`로 별도 보고합니다.
- **물리 work / NN total / raw branch / known-model force projection**을 구분합니다. 음의 shell 기여도 보존합니다.
- 원본 문서: [COMMON_PROTOCOL](../data/amb_kernel/specs/COMMON_PROTOCOL.md), [모델 목록](../data/amb_kernel/specs/README.md), [실행 안내](../data/amb_kernel/README.md).

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from datetime import datetime, timezone, timedelta
import json
import sys

# Repository root, repository parent, and notebooks/ are all supported.
HERE = Path.cwd().resolve()
candidates = [HERE, *HERE.parents, HERE / 'CNEEP_v2']
ROOT = next((p for p in candidates if (p / 'models/NEEP_ShellForce_2D.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run Jupyter inside CNEEP_v2 or its parent, or set ROOT explicitly.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from data.amb_kernel.model import AMBKernelModel, MODEL_NAMES, pilot_config, control_configs, laplacian
from data.amb_kernel.experiment import RunConfig, run_experiment, load_experiment
from data.amb_kernel.learning import (
    ArchitectureConfig, TrainingConfig, contiguous_splits, seed_splits,
    train_shell_force, load_checkpoint, evaluate_shell_force, projected_curl_check,
    source_provenance,
)

torch.set_num_threads(min(4, torch.get_num_threads()))
print('Repository:', ROOT)
print('Torch:', torch.__version__, '| simulation CPU/float64; optional training device below')

## 1. 한 곳에서 바꾸는 실험 설정

기존 notebook의 `H=64`, `n_hidden=3`, `R=3`, ELU를 여섯 모델 모두에 고정합니다. 외부 shell은 `learned_absolute`, center mixing 없음, exclusive Chebyshev mask입니다. `n_components=1`, `ep_component_indices=(0,)`로 추가 관측 채널을 만들지 않습니다.

첨부 명세의 기준 `shell_force.py` SHA256과 현재 저장소의 파일은 다릅니다. 아래에서 실제 hash와 생성자 계약을 기록합니다. 현재 구현의 scalar 입력은 `[B,2,Ny,Nx]`이므로 저장된 `[T,1,Ny,Nx]`에서 **singleton 축만** 제거합니다. 같은 소스라고 주장하지 않으며 네트워크 수정 없이 기존 옵션을 사용합니다.

In [ ]:
MODEL_IDS = ('M00', 'M01', 'M02', 'M03', 'M04', 'M05')
ARCHITECTURE = ArchitectureConfig(n_channel=64, n_hidden=3, max_distance=3,
                                  shell_force_activation='elu')
NN_METADATA = {**vars(ARCHITECTURE.options()),
               'constructor_configuration': 'fixed existing Corr_AMB2D_ShellForce options'}
QUICK_N = 12
QUICK_RUN = RunConfig(seed=73031, dt=1e-3, burn_in_steps=64,
                      measurement_steps=128, observation_stride=1,
                      observation_lags=(1, 4, 16), block_steps=32,
                      max_distance=ARCHITECTURE.max_distance)
# Override physical parameters here, separately for each model; defaults follow the specs.
MODEL_OVERRIDES = {model_id: {} for model_id in MODEL_IDS}
# Example: MODEL_OVERRIDES['M01'] = dict(g2=0.2, phi_bar=0.3)

stamp = datetime.now(timezone(timedelta(hours=9))).strftime('%Y%m%d-%H%M%S-%f')
OUTPUT = ROOT / 'results' / 'amb_kernel' / stamp
OUTPUT.mkdir(parents=True, exist_ok=False)
print('Output:', OUTPUT)
display(pd.DataFrame([{'model_id': key, 'description': MODEL_NAMES[key]} for key in MODEL_IDS]))
display(source_provenance())
display(vars(ARCHITECTURE.options()))
(OUTPUT / 'notebook_settings.json').write_text(json.dumps({
    'purpose': 'short CPU smoke; physical validation not established',
    'architecture': vars(ARCHITECTURE.options()),
    'quick_run': asdict(QUICK_RUN), 'physical_overrides': MODEL_OVERRIDES,
    'source_provenance': source_provenance(),
}, indent=2), encoding='utf8')

## 2. 모든 모델의 simulator 실행과 저장

매 integration step에서 fixed-noise implicit midpoint를 풀고 work와 drift-minus-trace를 누적합니다. solver 실패를 clipping, noise 재추출, 자동 timestep 변경으로 숨기지 않습니다. 실패하면 원인을 확인하고 해당 run을 더 작은 `dt`로 처음부터 다시 실행합니다.

각 run에는 `manifest.json`, `kernel.npz`, `fields.npy`, `times.npy`, `ep_blocks.csv`, `diagnostics.json` 및 block/observable 자료가 저장됩니다. Gaussian M03은 `b=0` calibration이며 M04의 `b=1`과 비교할 때는 `b`를 맞춘 별도 run이 필요합니다.

In [ ]:
quick_runs = {}
for model_id in MODEL_IDS:
    config = pilot_config(model_id, n=QUICK_N, **MODEL_OVERRIDES[model_id])
    result = run_experiment(config, QUICK_RUN, output_dir=OUTPUT / 'smoke' / model_id, nn_metadata=NN_METADATA)
    quick_runs[model_id] = result
    print(model_id, result.fields.shape, '| field mean drift:',
          float(np.max(np.abs(result.fields.mean(axis=(1, 2, 3)) - config.phi_bar))))
print('Six-model execution complete. These short trajectories are not physical validation.')

In [ ]:
physics_rows = []
for model_id, result in quick_runs.items():
    for row in result.ep_blocks:
        physics_rows.append({'model_id': model_id, **row})
physics_table = pd.DataFrame(physics_rows)
display(physics_table[(physics_table.mechanism == 'total') & (physics_table.shell == 'all')])
# Preserve the runner's actual stationarity, mass, tail and solver diagnostics.
display({model_id: result.diagnostics for model_id, result in quick_runs.items()})
physics_table.to_csv(OUTPUT / 'smoke_physics_table.csv', index=False)

## 3. Kernel, ??, stationarity ??

kernel ???? FFT shift? ?????. ????? ??? ??? `(0,0)`???. ?? ???? ? run? burn-in/block ??? ??? ?? ????, ?? ??? ??? ????? stationarity? ???? ????. ?? run? slow observable? `tau_int`, burn-in ? 20 tau, block ? 5 tau, ?? seed? ?????. Shell work ??? ??? ? block? ?? ?? ??????, ??? ?? ???? ??? block ? fluctuation? ?????.


In [ ]:
fig, axes = plt.subplots(3, len(quick_runs), figsize=(3 * len(quick_runs), 8))
odd_vmax = max(1e-12, max(float(np.max(np.abs(result.model.Ko))) for result in quick_runs.values()))
for col, (model_id, result) in enumerate(quick_runs.items()):
    model = result.model
    for row, values, title, cmap in (
        (0, np.fft.fftshift(model.K), f'{model_id}: full K', 'viridis'),
        (1, np.fft.fftshift(model.Ko), 'odd K', 'RdBu_r'),
        (2, result.fields[-1, 0], 'final phi (transient)', 'RdBu_r'),
    ):
        im = axes[row, col].imshow(values, origin='lower', cmap=cmap,
                                  **(dict(vmin=-odd_vmax, vmax=odd_vmax) if row == 1 else {}))
        axes[row, col].set_title(title)
        fig.colorbar(im, ax=axes[row, col], shrink=.6)
plt.tight_layout()
plt.savefig(OUTPUT / 'kernel_and_field.png', dpi=130)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for model_id, result in quick_runs.items():
    fields = result.fields[:, 0]
    variance = np.mean(fields ** 2, axis=(1, 2))
    grad_energy = .5 * np.mean(fields * laplacian(fields), axis=(1, 2))
    spectrum = np.abs(np.fft.fft2(fields)) ** 2 / result.model.N
    low_q = .5 * (spectrum[:, 0, 1] + spectrum[:, 1, 0])
    for ax, values in zip(axes, (variance, grad_energy, low_q)):
        ax.plot(result.times, values, label=model_id)
for ax, title in zip(axes, ('mean(phi^2)', 'gradient energy / site', 'selected low-q S')):
    ax.set(xlabel='physical time', title=title)
axes[0].legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'stationarity_observables.png', dpi=130)
plt.show()

In [ ]:
# Distribution across complete measurement blocks, rather than only the run mean.
fig, axes = plt.subplots(2, 3, figsize=(18, 10), squeeze=False)
for ax, (model_id, result) in zip(axes.flat, quick_runs.items()):
    active_mechanisms = {key.split(':')[0] for key in result.model.active_channels(result.fields[0, 0])}
    selected = [r for r in result.ep_blocks if r['method'] == 'midpoint'
                and r['mechanism'] in active_mechanisms
                and (r['shell'] == 'tail' or (str(r['shell']).startswith('r')
                     and int(r['shell'][1:]) <= ARCHITECTURE.max_distance))]
    mechanisms = sorted({r['mechanism'] for r in selected})
    labels = sorted({str(r['shell']) for r in selected},
                    key=lambda label: float('inf') if label == 'tail' else int(label[1:]))
    categories = [(mechanism, label) for mechanism in mechanisms for label in labels
                  if any(r['mechanism'] == mechanism and str(r['shell']) == label for r in selected)]
    positions = np.arange(len(categories))
    complete = result.block_step_counts == QUICK_RUN.block_steps
    datasets, ticklabels = [], []
    for mechanism, label in categories:
        key = f'midpoint|{mechanism}|{label}'
        values = np.asarray(result.block_values.get(key, []), dtype=float)
        values = values[complete[:len(values)]]
        datasets.append(values)
        ticklabels.append(f"{mechanism}\n{label.replace('tail', f'tail>{ARCHITECTURE.max_distance}')}")
    if datasets:
        ax.boxplot(datasets, positions=positions, widths=.62, showfliers=False,
                   patch_artist=True, boxprops={'facecolor':'#b9d7ea'},
                   medianprops={'color':'#174a6e','linewidth':1.5})
        for x, values in zip(positions, datasets):
            if len(values):
                jitter = np.linspace(-.10, .10, len(values)) if len(values) > 1 else np.zeros(1)
                ax.scatter(x + jitter, values, s=20, color='#e07a35', zorder=3, alpha=.8)
                ax.plot(x, np.mean(values), marker='D', color='black', ms=4, zorder=4)
    ax.set_xticks(positions, ticklabels, rotation=45, ha='right')
    ax.axhline(0, color='black', lw=.7)
    ax.set(title=model_id + ': shell work by block', ylabel='extensive EP rate')
    ax.grid(axis='y', alpha=.2)
fig.suptitle('Shell fluctuation across complete blocks (dots = blocks, diamond = mean)')
plt.tight_layout()
plt.savefig(OUTPUT / 'physical_shell_work_fluctuations.png', dpi=150)
plt.show()


## 4. 긴 실험: controls / independent seeds / dt / observation lag

`RUN_EXTENDED=True`로 선택한 군만 실행합니다. `SELECTED_CASE_NAMES=('all',)`은 해당 모델의 모든 control을 선택합니다. `control_configs`에는 M00 passive, source/receiver reciprocal·delta·uniform, M02 axis swap, M03 even/reflection, M04 full reciprocal·cancellation 등이 있습니다. `eta=1`만인 M04는 directional term이 남으므로 full equilibrium reference와 구분합니다.

아래 시간은 **출발 설정**입니다. 관측한 correlation time에 맞춰 늘려야 합니다. `dt/2`, `dt/4`에서도 같은 물리 burn-in/measurement/block 시간을 유지합니다. 기본 5 independent seeds를 사용하며 finite-size 비교는 `EXTENDED_N`을 32→64로 바꿔 별도로 실행합니다. grid refinement와 다릅니다. EPR 양성 판정에는 `sigma > 3 block_SE`뿐 아니라 stationarity 및 dt/lag 수렴이 필요합니다.

In [ ]:
RUN_EXTENDED = False
EXTENDED_MODEL_IDS = ('M00', 'M03', 'M01')
EXTENDED_N = 32
SELECTED_CASE_NAMES = ('primary',)  # e.g. ('primary', 'reciprocal') for M01; or ('all',)
SEEDS = (101, 102, 103, 104, 105)
DT_VALUES = (1e-3, 5e-4, 2.5e-4)
OBSERVATION_LAGS = (1, 4, 16)  # integration steps; record exact physical interval in each run
BURN_IN_TIME = 20.0
MEASUREMENT_TIME = 100.0
BLOCK_TIME = 1.0
OBSERVATION_STRIDE = 1  # must divide every selected observation lag

available_cases = {mid: control_configs(mid, n=EXTENDED_N, **MODEL_OVERRIDES[mid])
                   for mid in EXTENDED_MODEL_IDS}
display({mid: list(cases) for mid, cases in available_cases.items()})
extended_runs = {}  # run tag -> saved directory; do not retain all trajectories in RAM
extended_summary = []
frames_max = round(MEASUREMENT_TIME / min(DT_VALUES)) // OBSERVATION_STRIDE + 1
max_frame_gb = frames_max * EXTENDED_N ** 2 * 8 / 1e9
case_count = sum(len(cases) if SELECTED_CASE_NAMES == ('all',) else len(SELECTED_CASE_NAMES)
                 for cases in available_cases.values())
total_frame_gb = case_count * len(SEEDS) * sum(
    (round(MEASUREMENT_TIME / dt) // OBSERVATION_STRIDE + 1) * EXTENDED_N ** 2 * 8 / 1e9
    for dt in DT_VALUES)
print(f'Estimated float64 field storage: largest run {max_frame_gb:.2f} GB; total {total_frame_gb:.2f} GB.')
print('Peak RAM also includes working arrays/copies; tune durations and storage stride for your machine.')
if RUN_EXTENDED:
    for model_id, cases in available_cases.items():
        names = tuple(cases) if SELECTED_CASE_NAMES == ('all',) else SELECTED_CASE_NAMES
        for case_name in names:
            if case_name not in cases:
                raise ValueError(f'{model_id}: unknown case {case_name}; choose {list(cases)}')
            for dt in DT_VALUES:
                for seed in SEEDS:
                    run_config = RunConfig(
                        seed=seed, dt=dt, burn_in_steps=round(BURN_IN_TIME / dt),
                        measurement_steps=round(MEASUREMENT_TIME / dt),
                        observation_stride=OBSERVATION_STRIDE, observation_lags=OBSERVATION_LAGS,
                        block_steps=round(BLOCK_TIME / dt), max_distance=ARCHITECTURE.max_distance,
                    )
                    tag = f'{model_id}_{case_name}_dt{dt:g}_seed{seed}'
                    run_path = OUTPUT / 'extended' / tag
                    result = run_experiment(cases[case_name], run_config,
                                            output_dir=run_path, nn_metadata=NN_METADATA)
                    extended_runs[tag] = run_path
                    extended_summary.extend({'run_id': tag, **row} for row in result.ep_blocks
                                            if row['mechanism'] == 'total' and row['shell'] == 'all')
                    del result
                    print('Saved:', tag)
    pd.DataFrame(extended_summary).to_csv(OUTPUT / 'extended_summary.csv', index=False)
    display(pd.DataFrame(extended_summary))
else:
    print('Extended physics runs skipped; set RUN_EXTENDED=True after choosing cases and durations.')

## 5. 저장한 실험 재개와 finite-lag endpoint 확인

`load_experiment`로 기존 자료를 다시 읽습니다. fine integration-step streaming EPR과 관측 pair의 endpoint midpoint work는 다른 유한 시간간격 추정량입니다. 아래 표는 차이를 직접 보여 줍니다. 동일 physical lag끼리 비교하려면 `lag_steps * dt`도 맞추세요.

In [ ]:
SAVED_RUN = None  # e.g. ROOT / 'results/amb_kernel/.../smoke/M01'
if SAVED_RUN is not None:
    restored = load_experiment(SAVED_RUN)
    print('Restored:', restored.model.config.model_id, restored.fields.shape)

lag_rows = []
for model_id, result in quick_runs.items():
    x = result.fields[:, 0]
    frame_dt = float(np.diff(result.times).mean())
    for lag in (1, 4, 16):
        work = []
        for i in range(0, len(x) - lag, lag):
            mid = .5 * (x[i] + x[i + lag])
            channels = result.model.active_channels(mid)
            active = sum(channels.values(), np.zeros_like(mid))
            work.append(-np.sum(active * (x[i + lag] - x[i])) / result.model.config.D)
        lag_rows.append({'model_id': model_id, 'lag_frames': lag,
                         'physical_lag': lag * frame_dt,
                         'endpoint_rate': np.mean(work) / (lag * frame_dt), 'n_intervals': len(work)})
lag_table = pd.DataFrame(lag_rows)
display(lag_table)
lag_table.to_csv(OUTPUT / 'finite_lag_endpoint_work.csv', index=False)

## 6. ?? ??? ?? / ??

? ?? ??? trajectory? ??? train/validation/test split? ?????. `TRAIN_SPLIT='contiguous'`? ?? ??? `TRAINING_RUNS`? ??? ???? ???, `independent_seeds`? ?? seed? ?? ?????? ?????. ??? ?? ??? ?? ?? ?? ???? ???.


In [ ]:
MODELS_TO_TRAIN = MODEL_IDS  # e.g. ('M03',)
TRAIN_SPLIT = 'contiguous'  # or 'independent_seeds'
TRAINING_RUNS = quick_runs  # ? ?? run ?? ? ?? ??
TRAIN_SEEDS, VALIDATION_SEEDS, TEST_SEEDS = (201, 202, 203), (204,), (205,)
training_splits = {}

for model_id in MODELS_TO_TRAIN:
    reference = TRAINING_RUNS[model_id]
    if TRAIN_SPLIT == 'contiguous':
        splits = contiguous_splits(reference.fields)
    elif TRAIN_SPLIT == 'independent_seeds':
        by_seed = {}
        run_settings = reference.manifest['run_config']
        run_fields = set(RunConfig.__dataclass_fields__)
        base = RunConfig(**{k: v for k, v in run_settings.items() if k in run_fields})
        for seed in TRAIN_SEEDS + VALIDATION_SEEDS + TEST_SEEDS:
            run_path = OUTPUT / 'training_data' / model_id / f'seed{seed}'
            independent = run_experiment(reference.model, replace(base, seed=seed),
                output_dir=run_path, nn_metadata=NN_METADATA)
            by_seed[seed] = independent.fields.copy()
            print('Saved training trajectory:', run_path)
        splits = seed_splits(by_seed, TRAIN_SEEDS, VALIDATION_SEEDS, TEST_SEEDS)
    else:
        raise ValueError("TRAIN_SPLIT must be 'contiguous' or 'independent_seeds'.")
    training_splits[model_id] = splits
    split_path = OUTPUT / 'training_data' / model_id / 'splits.npz'
    split_path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {f'{name}_{i}': value for name in ('train', 'validation', 'test')
              for i, value in enumerate(getattr(splits, name))}
    index = {name: len(getattr(splits, name)) for name in ('train', 'validation', 'test')}
    np.savez_compressed(split_path, **arrays, index=json.dumps(index),
                        metadata=json.dumps(splits.metadata))
    print(model_id, splits.metadata, '| split archive:', split_path)



## 7. ?? ??

? ??? ?? ?? ?? ??? ? ? ??? ?? ??? ???? ?????. ? ?? ??????? split ??? ?? ????.


In [ ]:
TRAINING = TrainingConfig(iterations=300, batch_size=32, learning_rate=1e-3,
                          alpha=-0.5, record_every=25, seed=3,
                          device='cpu', dtype='float64', lag_frames=1)
# Use 'cuda' above when desired; the identical network/physical units are retained.
trained, evaluations = {}, {}

for model_id in MODELS_TO_TRAIN:
    if model_id not in training_splits:
        raise RuntimeError(f'Missing prepared data for {model_id}; run the dataset preparation cell first.')
    reference = TRAINING_RUNS[model_id]
    splits = training_splits[model_id]
    fit = train_shell_force(splits, OUTPUT / 'learning' / model_id,
                            architecture=ARCHITECTURE, config=TRAINING,
                            physical_metadata=reference.manifest)
    trained[model_id] = fit
    reloaded = load_checkpoint(fit.checkpoint, device=TRAINING.device)
    frame_dt = float(np.diff(reference.times).mean())
    evaluations[model_id] = evaluate_shell_force(
        reloaded, splits.test, reference.model, frame_dt,
        lag_frames=TRAINING.lag_frames, batch_size=TRAINING.batch_size,
        block_pairs=16, output_dir=OUTPUT / 'learning' / model_id)
    print(model_id, 'best checkpoint:', fit.checkpoint)
    display(pd.DataFrame(evaluations[model_id]['rows']))


## 8. NN total / raw branch / known-force projection / curl

- `NN_total`: N배 공간 평균 score를 observation interval로 나눈 extensive rate.
- `endpoint_midpoint`: 동일 test pair에서 알려진 active work. streaming truth와 별도입니다.
- `NN_raw_branch`: network가 선택한 additive decomposition; 물리 채널과 같다고 가정하지 않습니다.
- `NN_force_projection`: **−〈nu_cᵀ L d_theta,total〉**, 왼쪽 endpoint의 equal-time 진단. 이미 rate이므로 lag로 다시 나누지 않습니다.
- projected curl은 zero-mean 방향의 bilinear test로 `J_d−J_dᵀ = −(J_mu−J_muᵀ)/D`를 검사합니다. total curl만으로 branch attribution이나 global exactness를 증명하지 않습니다.

block SE는 인접 pair를 독립 표본으로 세는 naive SEM이 아닙니다. 선택한 block 길이가 tau에 비해 충분한지 따로 확인해야 합니다. M05의 `u*u_r`는 attribution convention이며 `r=0` physical channel은 다른 shell 정보에도 의존하므로 NN on-site null과 다릅니다.

In [ ]:
if trained:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for model_id, fit in trained.items():
        history = pd.DataFrame(fit.history)
        axes[0].plot(history.iteration, history.validation_loss, label=model_id)
        rows = evaluations[model_id]['rows']
        raw = [r for r in rows if r['method'] == 'NN_raw_branch']
        axes[1].plot(range(len(raw)), [r['mean'] for r in raw], marker='o', label=model_id)
        phi = training_splits[model_id].test[0][0, 0]
        curl = projected_curl_check(fit, TRAINING_RUNS[model_id].model, phi, probes=8)
        display({'model_id': model_id, 'projected_total_curl': curl})
        (OUTPUT / 'learning' / model_id / 'projected_curl.json').write_text(
            json.dumps(curl, indent=2), encoding='utf8')
    axes[0].set(xlabel='iteration', ylabel='joint extensive validation loss')
    axes[1].set(xlabel='Chebyshev shell r', ylabel='signed NN raw extensive EP rate')
    axes[1].axhline(0, color='black', lw=.7)
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / 'learning_curves_and_shells.png', dpi=130)
    plt.show()

In [ ]:
if trained:
    model_id = next(iter(trained))
    fit, reference = trained[model_id], TRAINING_RUNS[model_id]
    test = training_splits[model_id].test[0]
    lag = TRAINING.lag_frames
    pair_phi = np.stack((test[0, 0], test[lag, 0]))
    parameter = next(fit.model.parameters())
    z = fit.normalizer.transform(torch.as_tensor(pair_phi, dtype=torch.float64)[None])
    with torch.no_grad():
        maps = fit.model(z.to(parameter.device, parameter.dtype), return_maps=True)[0].cpu().numpy()
    delta_t = lag * float(np.diff(reference.times).mean())
    pred_local_rate = maps.sum(axis=0) / delta_t
    midpoint, delta = pair_phi.mean(axis=0), pair_phi[1] - pair_phi[0]
    channels = reference.model.active_channels(midpoint)
    known_active = sum(channels.values(), np.zeros_like(midpoint))
    truth_local_rate = -known_active * delta / (reference.model.config.D * delta_t)
    vmax = max(np.max(np.abs(pred_local_rate)), np.max(np.abs(truth_local_rate)), 1e-12)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, values, title in zip(axes, (midpoint, truth_local_rate, pred_local_rate),
                                 ('phi midpoint', 'known endpoint work / site / time', 'NN local rate')):
        image = ax.imshow(values, origin='lower', cmap='RdBu_r',
                          **({} if title == 'phi midpoint' else dict(vmin=-vmax, vmax=vmax)))
        ax.set_title(model_id + ': ' + title)
        fig.colorbar(image, ax=ax, shrink=.7)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'{model_id}_local_ep_comparison.png', dpi=130)
    plt.show()
    print('Spatial SUM of each rate map gives its extensive total; no additional N factor.')

## 결과를 해석하기 전

1. 원본 명세의 algebra checks와 simulator의 noise/mass/kernel/trace tests를 통과했는지 확인합니다.
2. passive/reciprocal/null controls, stationarity, autocorrelation 기반 burn-in·block, independent seeds, dt/lag 수렴을 검토합니다. Gaussian M03에서는 analytic covariance/EPR도 비교합니다.
3. full kernel tail, 모델별 표현 한계, 총량과 밀도, 유한 관측 간격을 구분한 뒤 NN 결과를 비교합니다. H/R을 모델별로 조절해 결과를 맞추지 않습니다.
4. 총 EPR 일치와 raw branch attribution 일치는 별개입니다. source quadratic-rank 및 receiver cross-shell 의존성을 남은 오차의 가능한 원인으로 기록합니다.

모든 그림과 표는 이 notebook 실행의 `OUTPUT` 아래에 저장됩니다. checkpoint는 `learning/<model>/best_model_parameter.pth.tar`, 생성자·정규화·loss·split·source hash는 `learning_manifest.json`에서 확인할 수 있습니다.